# E-Commerce Data Pipeline - Analytics & Business Insights

This interactive notebook explores synthetic e-commerce datasets generated by `SyntheticDataGenerator`:
- **Customers**: Normal distribution for `age` centered around 35.
- **Products**: Categories (`Electronics`, `Clothing`, `Home`, `Sports`, `Books`), `price` ($10-$500), `rating` (1-5).
- **Orders**: Pareto distribution (80/20 rule: 20% of customers make ~80% of orders), `quantity` (1-10).

In [ ]:
import sys
from pathlib import Path

# Ensure project root is in sys.path
project_root = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import pandas as pd
import numpy as np
from src import config

print(f"Project root: {project_root}")
print(f"Raw data directory: {config.DATA_ROW_DIR}")
print(f"Results directory: {config.DATA_PROCEED_DIR}")

## 1. Load Raw Data (Pandas)

In [ ]:
customers_df = pd.read_csv(config.CUSTOMERS_FILE)
products_df = pd.read_csv(config.PRODUCTS_FILE)
orders_df = pd.read_csv(config.ORDERS_FILE)

print(f"Loaded {len(customers_df):,} customers")
print(f"Loaded {len(products_df):,} products")
print(f"Loaded {len(orders_df):,} orders")

customers_df.head(3)

## 2. Verify Customer Age Distribution (Normal around 35)

In [ ]:
age_stats = customers_df["age"].describe()
print("Customer Age Distribution:")
print(age_stats.round(2))
print(f"Mean age: {customers_df['age'].mean():.2f} (Target ~ 35)")

## 3. Verify Product Catalog Constraints

In [ ]:
prod_cat = products_df.groupby("category").agg(
    count=("product_id", "count"),
    min_price=("price", "min"),
    avg_price=("price", "mean"),
    max_price=("price", "max"),
    avg_rating=("rating", "mean")
).round(2).reset_index()
prod_cat

## 4. Verify Pareto 80/20 Distribution in Orders

In [ ]:
order_counts = orders_df["customer_id"].value_counts().values
total_customers = len(customers_df)
top_20_count = int(0.20 * total_customers)
top_20_orders = order_counts[:top_20_count].sum()
pct_share = (top_20_orders / len(orders_df)) * 100.0

print(f"Total unique customers: {total_customers:,}")
print(f"Top 20% customers count: {top_20_count:,}")
print(f"Orders by top 20% customers: {top_20_orders:,} out of {len(orders_df):,}")
print(f"Empirical Pareto share: {pct_share:.2f}% of orders")

## 5. PySpark Analytics Engine

In [ ]:
from src.spark_analyzer import create_spark_session, load_raw_data, analyze_category_performance, analyze_customer_lifetime_value

spark = create_spark_session()
datasets = load_raw_data(spark)

cat_perf = analyze_category_performance(datasets["products"], datasets["orders"])
cat_perf.show(truncate=False)

spark.stop()